In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

In [ ]:
path = r"data\dataset.csv"
with open(path) as file:
    df = pd.read_csv(file)
df.head()

In [3]:
# Training
# selecting features and target
features = df[['year', 'shootouts', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y', 'home_wc_titles_before',
       'home_total_market_value_eur', 'home_avg_age',
       'home_wc_participations_before', 'home_groups_passed_before',
       'home_round16_before', 'home_quarterfinals_before',
       'home_semifinals_before', 'home_finals_before',
       'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y', 'away_wc_titles_before',
       'away_total_market_value_eur', 'away_avg_age',
       'away_wc_participations_before', 'away_groups_passed_before',
       'away_round16_before', 'away_quarterfinals_before',
       'away_semifinals_before', 'away_finals_before']] 
target = df[['year', 'result']]

In [4]:
# Using Logistic regression 
# Accuracy and loss are computed on the train set

# to be modified: now it outputs a binary variable (it is totally wrong), I should implement a softmax regression with 3 classes
# try a ridge regularization and see what happens, first it is essential to standardize continous variables

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target.loc[target['year'] != 2026, 'result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target.loc[target['year'] == 2026, 'result']

model = LogisticRegression(max_iter=200)
_ = model.fit(X_train, Y_train)

prob = _.predict_proba(X_train)
preds = _.predict(X_train)

acc = accuracy_score(Y_train, preds)
loss = log_loss(Y_train, prob)

print(f"World Cup 2006-2022: Accuracy: {acc:.4f}, LogLoss: {loss:.4f}")

World Cup 2006-2022: Accuracy: 0.5969, LogLoss: 0.9466


In [5]:
# computing accuracy and loss on test set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result']

model = LogisticRegression(max_iter=200)
_ = model.fit(X_train, Y_train)

prob = _.predict_proba(X_test)
preds = _.predict(X_test)

acc = accuracy_score(Y_test, preds)
loss = log_loss(Y_test, prob)

print(f"World Cup 2026: Accuracy: {acc:.4f}, LogLoss: {loss:.4f}")

World Cup 2026: Accuracy: 0.7010, LogLoss: 0.8884


In [6]:
# Trying XGBoost
# Accuracy and loss are computed on the training set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result']

model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3,
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=32, n_estimators=10,
        eval_metric="mlogloss", random_state=42,
)
model.fit(X_train, Y_train)

prob = model.predict_proba(X_train)
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])
df_prob["stage"] = df[df["year"] < 2026]["stage"].values

# compute normalization only for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    if row["stage"] == 1:
        return [row["home_ko_prob"], row["away_ko_prob"]]
    else:
        return [row["home_prob"], row["draw_prob"], row["away_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

def predict(row):
    if row["stage"] == 1:
        if row["home_ko_prob"] > row["away_ko_prob"]:
            return 0
        else:
            return 1
    else:
        classes = [0, 1, 2]
        groups_prob = [row["home_prob"], row["away_prob"], row["draw_prob"]]
        return classes[np.argmax(groups_prob)]

df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

acc = accuracy_score(Y_train, preds)
# loss = log_loss(Y_train, prob)

print(f"World Cup 2006-2022: Accuracy: {acc:.4f}") #| LogLoss: {loss:.4f}")

World Cup 2006-2022: Accuracy: 0.8063


In [7]:
# Computing accuracy and loss on the test set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result']

model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3,
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=32.0, n_estimators=10,
        eval_metric="mlogloss", random_state=42,
)
model.fit(X_train, Y_train)

prob = model.predict_proba(X_test)
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])
df_prob["stage"] = df[df["year"] == 2026]["stage"].values

# compute normalization only for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    if row["stage"] == 1:
        return [row["home_ko_prob"], row["away_ko_prob"]]
    else:
        return [row["home_prob"], row["draw_prob"], row["away_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

def predict(row):
    if row["stage"] == 1:
        if row["home_ko_prob"] > row["away_ko_prob"]:
            return 0
        else:
            return 1
    else:
        classes = [0, 1, 2]
        groups_prob = [row["home_prob"], row["away_prob"], row["draw_prob"]]
        return classes[np.argmax(groups_prob)]
    
df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

acc = accuracy_score(Y_test, preds)
# loss = log_loss(Y_test, prob, labels=[0, 1, 2])

print(f"World Cup 2026: Accuracy: {acc:.4f}") # | LogLoss: {loss:.4f}")

# To be done: implementing a Random Forest
# Computing and adding the Elo and / or Fifa ranking 

World Cup 2026: Accuracy: 0.8557
